# 07 — XGBoost v2 (Tight Tuning with Fold-Level Early Stopping)

**Why this notebook exists.** The first XGBoost pass (notebook 04) ended at CV MSE ≈ 222.8 — worse than GradientBoosting (183.8). Root causes:
1. `RandomizedSearchCV` with no early stopping wasted budget on under-trained configs.
2. Tuning was 3-fold but final eval was 5-fold (slight mismatch).
3. The chosen `learning_rate ≈ 0.016` × `n_estimators = 977` ⇒ effective shrinkage too low, model under-converged.

**Strategy here.**
* Hand-rolled 5-fold CV — same seed as everything else (so OOFs stay blendable).
* Inside each fold, split a 10% inner validation set for **`early_stopping_rounds=50`**.
* Random search over **60 configurations** with a focused, well-defined parameter space.
* `log1p` target + `KFoldTargetEncoder` (Module-7 custom transformer) reused.
* For each config we report mean **fold-best-MSE** ± std → pick the lowest mean.
* Refit on full train with the best config + final OOF.

Outputs:
* `outputs/oof_XGBoostV2.npy` — replaces `oof_XGBoost.npy` in the blend
* `outputs/submission_xgboost_v2.csv`
* `outputs/xgb_v2_best_params.json`
* `outputs/xgb_v2_search_log.csv` — every config + score, for the paper.

All techniques are course-allowed: XGBoost + RandomizedSearch + KFold CV + Pipeline + custom transformer + early stopping (Module 10/11).

## 1. Setup & Data

In [1]:
import warnings, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
N_SPLITS = 5

OUT_DIR = Path('../outputs'); OUT_DIR.mkdir(exist_ok=True, parents=True)
print('OUT_DIR =', OUT_DIR.resolve())

OUT_DIR = /Users/bashkal/Desktop/Comp468-ML_in_Python/ali/ML-Final/outputs


In [2]:
train_df = pd.read_parquet(OUT_DIR / 'train_features.parquet')
test_df  = pd.read_parquet(OUT_DIR / 'test_features.parquet')
TARGET = 'NumReserveDays2016Q3'; ID = 'PropertyID'
y = train_df[TARGET].astype(float).values
X = train_df.drop(columns=[TARGET, ID]).reset_index(drop=True)
X_test = test_df.drop(columns=[ID]).reindex(columns=X.columns).reset_index(drop=True)
test_ids = test_df[ID].values

cat_all = X.select_dtypes(exclude='number').columns.tolist()
card    = {c: X[c].nunique(dropna=False) for c in cat_all}
cat_high = [c for c, n in card.items() if n > 15]
cat_low  = [c for c, n in card.items() if n <= 15]
num_cols = X.select_dtypes(include='number').columns.tolist()
print(f'num={len(num_cols)}  cat_low={cat_low}  cat_high={cat_high}')

num=143  cat_low=['ListingType', 'MetropolitanStatisticalArea', 'CancellationPolicy', 'geo_cluster']  cat_high=['PropertyType', 'Neighborhood']


## 2. K-Fold Target Encoder (Module-7 custom transformer)

In [3]:
class KFoldTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols, n_splits=5, smoothing=20.0, random_state=42):
        self.cols = cols; self.n_splits = n_splits
        self.smoothing = smoothing; self.random_state = random_state

    def _smap(self, x, y):
        st = pd.DataFrame({'c': x, 'y': y}).groupby('c')['y'].agg(['mean', 'count'])
        return ((st['count'] * st['mean'] + self.smoothing * self.global_mean_)
                / (st['count'] + self.smoothing)).to_dict()

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        self.global_mean_ = float(y.mean())
        self.maps_ = {c: self._smap(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return self

    def transform(self, X):
        Xo = X.copy()
        for c in self.cols:
            Xo[c] = (X[c].astype(str).fillna('__nan__').map(self.maps_[c])
                       .fillna(self.global_mean_).astype('float32'))
        return Xo

    def fit_transform(self, X, y=None, **kw):
        y = np.asarray(y, dtype=float); self.global_mean_ = float(y.mean())
        Xo = X.copy()
        for c in self.cols: Xo[c] = np.full(len(X), self.global_mean_, dtype='float32')
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        for tr, va in kf.split(X):
            for c in self.cols:
                m = self._smap(X[c].astype(str).fillna('__nan__').iloc[tr], y[tr])
                Xo.iloc[va, Xo.columns.get_loc(c)] = (
                    X[c].astype(str).fillna('__nan__').iloc[va].map(m)
                       .fillna(self.global_mean_).astype('float32').values)
        self.maps_ = {c: self._smap(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return Xo

    def get_feature_names_out(self, input_features=None):
        return np.array(input_features if input_features is not None else self.cols)

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
        ('low', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
            ('oh',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_low),
        ('high', Pipeline([
            ('te', KFoldTargetEncoder(cols=cat_high, n_splits=5, smoothing=20, random_state=RANDOM_STATE)),
        ]), cat_high),
    ])

## 3. Fold-Level Early-Stopping Evaluator
Each candidate hyperparameter config is evaluated like this:
1. Run a 5-fold CV with the *same* outer split as our other notebooks.
2. **Inside each fold:** carve a 10% inner-validation slice from the training portion. Fit XGBoost with `early_stopping_rounds=50` watching that inner slice. Use the resulting model to predict the outer-val slice → fold MSE.
3. Return mean ± std over folds.
This makes `n_estimators` adaptive: each fold picks the best iteration on its own.

In [4]:
outer_kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
outer_splits = list(outer_kf.split(X))   # cache so all configs see the same folds

def evaluate_config(params, return_oof=False, return_models=False, verbose=False):
    """Run 5-fold CV with fold-level early stopping. Returns mean MSE, std, best_iters.
    Optionally returns OOF predictions and the trained models for later refit/inference."""
    fold_mses, best_iters, models = [], [], []
    oof = np.zeros(len(y)) if return_oof else None

    for fold, (tr, va) in enumerate(outer_splits):
        pp = make_preprocessor()
        X_tr_full = pp.fit_transform(X.iloc[tr], y[tr])
        X_va      = pp.transform(X.iloc[va])

        # inner split for early stopping
        X_tr, X_in, y_tr, y_in = train_test_split(
            X_tr_full, y[tr], test_size=0.10, random_state=RANDOM_STATE + fold,
        )
        y_tr_log = np.log1p(y_tr); y_in_log = np.log1p(y_in)

        model = XGBRegressor(
            **params,
            objective='reg:squarederror',
            tree_method='hist',
            random_state=RANDOM_STATE + fold,
            n_jobs=-1,
            early_stopping_rounds=50,
            eval_metric='rmse',
        )
        model.fit(X_tr, y_tr_log, eval_set=[(X_in, y_in_log)], verbose=False)
        best_iter = model.best_iteration

        # predict outer val, invert log1p, clip to [0, 92]
        pred = np.clip(np.expm1(model.predict(X_va, iteration_range=(0, best_iter + 1))), 0, 92)
        mse  = mean_squared_error(y[va], pred)

        fold_mses.append(mse); best_iters.append(best_iter)
        if return_oof: oof[va] = pred
        if return_models: models.append((model, pp, best_iter))
        if verbose: print(f'    fold {fold+1}: MSE={mse:.3f}, best_iter={best_iter}')

    return {
        'mse_mean': float(np.mean(fold_mses)),
        'mse_std':  float(np.std(fold_mses)),
        'best_iters': best_iters,
        'fold_mses': fold_mses,
        'oof': oof,
        'models': models,
    }

## 4. Sanity-Check Config (Sensible Defaults)
Confirms the pipeline works and gives us a sane baseline before any search.

In [5]:
default_cfg = dict(
    n_estimators=3000,           # large; early stopping picks the real count
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=4,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
)
t0 = time.time()
res0 = evaluate_config(default_cfg, verbose=True)
print(f"\nDefault XGB | MSE = {res0['mse_mean']:.3f} ± {res0['mse_std']:.3f} "
      f"| iters = {res0['best_iters']} | {time.time()-t0:.1f}s")

    fold 1: MSE=215.894, best_iter=186
    fold 2: MSE=238.231, best_iter=220
    fold 3: MSE=227.772, best_iter=198
    fold 4: MSE=220.534, best_iter=161
    fold 5: MSE=223.366, best_iter=180

Default XGB | MSE = 225.159 ± 7.592 | iters = [186, 220, 198, 161, 180] | 9.1s


## 5. Random Search — 60 configurations
Search space focused on the ranges that matter for tabular boosting. Learning rates are sampled log-uniform but biased toward the practical 0.03-0.08 sweet spot.

In [6]:
rng = np.random.default_rng(RANDOM_STATE)

def sample_config():
    return dict(
        n_estimators     = 3000,   # cap; early-stop will trim
        learning_rate    = float(np.exp(rng.uniform(np.log(0.02), np.log(0.10)))),
        max_depth        = int(rng.integers(4, 9)),       # 4..8
        min_child_weight = int(rng.integers(1, 12)),      # 1..11
        subsample        = float(rng.uniform(0.65, 0.95)),
        colsample_bytree = float(rng.uniform(0.65, 0.95)),
        gamma            = float(rng.uniform(0.0, 0.4)),
        reg_alpha        = float(np.exp(rng.uniform(np.log(1e-3), np.log(0.5)))),
        reg_lambda       = float(np.exp(rng.uniform(np.log(0.5),  np.log(5.0)))),
    )

N_ITER = 60
log_rows = []
best_score = res0['mse_mean']
best_cfg   = default_cfg.copy()

t_search = time.time()
for i in range(N_ITER):
    cfg = sample_config()
    t0 = time.time()
    res = evaluate_config(cfg)
    elapsed = time.time() - t0
    log_rows.append({**cfg, 'mse_mean': res['mse_mean'], 'mse_std': res['mse_std'],
                     'avg_best_iter': int(np.mean(res['best_iters'])), 'seconds': elapsed})
    flag = ''
    if res['mse_mean'] < best_score:
        best_score = res['mse_mean']
        best_cfg   = cfg.copy()
        flag = '  ←  NEW BEST'
    print(f"[{i+1:2d}/{N_ITER}] MSE = {res['mse_mean']:7.3f} ± {res['mse_std']:5.3f} "
          f"| iters≈{int(np.mean(res['best_iters'])):4d} | lr={cfg['learning_rate']:.4f} "
          f"depth={cfg['max_depth']} | {elapsed:5.1f}s{flag}")

print(f'\nSearch finished in {(time.time()-t_search)/60:.1f} min')
print(f'Best CV MSE: {best_score:.3f}')
print('Best config:', best_cfg)

log_df = pd.DataFrame(log_rows).sort_values('mse_mean').reset_index(drop=True)
log_df.to_csv(OUT_DIR / 'xgb_v2_search_log.csv', index=False)
with open(OUT_DIR / 'xgb_v2_best_params.json', 'w') as f:
    json.dump(best_cfg, f, indent=2)
log_df.head(10)

[ 1/60] MSE = 226.227 ± 6.717 | iters≈ 169 | lr=0.0695 depth=7 |  10.5s
[ 2/60] MSE = 229.097 ± 6.427 | iters≈ 113 | lr=0.0709 depth=6 |   6.3s
[ 3/60] MSE = 226.611 ± 8.651 | iters≈ 197 | lr=0.0408 depth=6 |   9.5s
[ 4/60] MSE = 231.826 ± 7.084 | iters≈ 537 | lr=0.0354 depth=4 |  12.0s
[ 5/60] MSE = 225.432 ± 5.839 | iters≈ 327 | lr=0.0256 depth=7 |  16.7s
[ 6/60] MSE = 225.773 ± 7.529 | iters≈ 310 | lr=0.0271 depth=6 |  12.4s
[ 7/60] MSE = 231.698 ± 7.978 | iters≈ 277 | lr=0.0617 depth=4 |   7.3s
[ 8/60] MSE = 226.754 ± 5.349 | iters≈ 274 | lr=0.0250 depth=8 |  18.8s
[ 9/60] MSE = 226.265 ± 8.117 | iters≈ 212 | lr=0.0419 depth=6 |   7.9s
[10/60] MSE = 226.477 ± 7.059 | iters≈ 131 | lr=0.0685 depth=6 |   6.0s
[11/60] MSE = 227.799 ± 7.117 | iters≈ 444 | lr=0.0282 depth=5 |  11.3s
[12/60] MSE = 232.867 ± 8.356 | iters≈ 255 | lr=0.0580 depth=4 |   6.8s
[13/60] MSE = 231.821 ± 8.079 | iters≈ 695 | lr=0.0207 depth=4 |  15.0s
[14/60] MSE = 227.045 ± 6.876 | iters≈ 163 | lr=0.0613 depth=6 |

,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,mse_mean,mse_std,avg_best_iter,seconds
0,3000,0.020582,7,2,0.898829,0.889045,0.093056,0.027073,2.018300,223.231848,6.615554,486,23.882528
1,3000,0.023819,7,8,0.734370,0.847827,0.290798,0.118729,0.640783,223.793270,7.075195,323,14.312072
2,3000,0.043771,7,6,0.931348,0.821518,0.189396,0.005255,1.072850,223.930110,6.499194,230,10.804793
3,3000,0.023308,7,8,0.675348,0.930782,0.054963,0.387248,3.161216,224.256825,7.206133,344,16.023969
4,3000,0.031584,7,10,0.681926,0.949731,0.266274,0.056842,0.615759,224.315840,6.131045,242,11.843717
5,3000,0.030912,7,5,0.744332,0.697283,0.059113,0.336186,1.370484,224.552497,7.973760,271,12.600530
6,3000,0.032725,8,7,0.703032,0.906984,0.303408,0.087460,1.352269,224.701383,6.840892,237,15.080555
7,3000,0.034009,7,2,0.681021,0.826293,0.068237,0.313958,1.905597,224.790312,6.758053,207,11.401349
8,3000,0.024836,8,6,0.804267,0.907272,0.185120,0.010948,2.180385,224.816195,7.189076,331,21.104403
9,3000,0.034953,7,7,0.656841,0.937568,0.192921,0.129592,0.604923,224.894785,6.103920,244,12.154448


## 6. Final 5-Fold OOF with Best Config
Re-evaluate the winner to also extract OOF predictions and the per-fold trained models (for the test-set inference).

In [7]:
final_res = evaluate_config(best_cfg, return_oof=True, return_models=True, verbose=True)
print(f"\nXGBoost v2 | OOF MSE = {final_res['mse_mean']:.3f} ± {final_res['mse_std']:.3f}")
np.save(OUT_DIR / 'oof_XGBoostV2.npy', final_res['oof'])

    fold 1: MSE=214.545, best_iter=431
    fold 2: MSE=233.208, best_iter=648
    fold 3: MSE=226.613, best_iter=481
    fold 4: MSE=217.618, best_iter=300
    fold 5: MSE=224.176, best_iter=573

XGBoost v2 | OOF MSE = 223.232 ± 6.616


## 7. Test-Set Prediction — Average the 5 Fold-Models
Each fold's model used early stopping → averaging them is more robust than refitting on full data with a fixed iter count.

In [8]:
test_preds = np.zeros(len(X_test))
for model, pp, best_iter in final_res['models']:
    X_test_proc = pp.transform(X_test)
    p = np.expm1(model.predict(X_test_proc, iteration_range=(0, best_iter + 1)))
    test_preds += np.clip(p, 0, 92)
test_preds /= len(final_res['models'])
test_preds_int = np.clip(np.round(test_preds), 0, 92).astype(int)
print('Test preds:', test_preds_int.min(), '→', test_preds_int.max(),
      '| mean =', test_preds_int.mean().round(2))

Test preds: 0 → 90 | mean = 13.73


In [9]:
# Write submission with the byte-safe pattern that worked for Kaggle
lines = [b'PropertyID_test,Pred\n']
for i, p in zip(test_ids, test_preds_int):
    lines.append(f'{int(i)},{int(p)}\n'.encode('ascii'))
(OUT_DIR / 'submission_xgboost_v2.csv').write_bytes(b''.join(lines))

chk = pd.read_csv(OUT_DIR / 'submission_xgboost_v2.csv')
print('Rows:', len(chk), '| NaN:', chk.isna().sum().sum(), '| header:', list(chk.columns))
chk.head()

Rows: 24318 | NaN: 0 | header: ['PropertyID_test', 'Pred']


,PropertyID_test,Pred
0,795,0
1,2515,43
2,2595,2
3,5099,34
4,5107,17


## 8. Top-20 Search Configs (for the paper)

In [10]:
show_cols = ['mse_mean', 'mse_std', 'learning_rate', 'max_depth', 'min_child_weight',
             'subsample', 'colsample_bytree', 'gamma', 'reg_alpha', 'reg_lambda',
             'avg_best_iter', 'seconds']
log_df.head(20)[show_cols]

,mse_mean,mse_std,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,avg_best_iter,seconds
0,223.231848,6.615554,0.020582,7,2,0.898829,0.889045,0.093056,0.027073,2.018300,486,23.882528
1,223.793270,7.075195,0.023819,7,8,0.734370,0.847827,0.290798,0.118729,0.640783,323,14.312072
2,223.930110,6.499194,0.043771,7,6,0.931348,0.821518,0.189396,0.005255,1.072850,230,10.804793
3,224.256825,7.206133,0.023308,7,8,0.675348,0.930782,0.054963,0.387248,3.161216,344,16.023969
4,224.315840,6.131045,0.031584,7,10,0.681926,0.949731,0.266274,0.056842,0.615759,242,11.843717
5,224.552497,7.973760,0.030912,7,5,0.744332,0.697283,0.059113,0.336186,1.370484,271,12.600530
6,224.701383,6.840892,0.032725,8,7,0.703032,0.906984,0.303408,0.087460,1.352269,237,15.080555
7,224.790312,6.758053,0.034009,7,2,0.681021,0.826293,0.068237,0.313958,1.905597,207,11.401349
8,224.816195,7.189076,0.024836,8,6,0.804267,0.907272,0.185120,0.010948,2.180385,331,21.104403
9,224.894785,6.103920,0.034953,7,7,0.656841,0.937568,0.192921,0.129592,0.604923,244,12.154448


## Summary
- Hand-rolled CV with **fold-level early stopping** gave each config a fair shot.
- 60 random configs explored, best logged to `xgb_v2_search_log.csv`.
- New OOF saved as `oof_XGBoostV2.npy` — drop into notebook 06 to rebuild the blend.
- New test submission `submission_xgboost_v2.csv` ready (header `PropertyID_test,Pred`, integers, byte-safe writer).

Next step: re-run notebook **06** to rebuild the blend with the improved XGBoost OOF.